# NUMPY MASTERY — CHAPTERS 15–17

```
NUMPY MASTERY
Current Chapters: 15 (NaN and Infinity), 16 (Sorting and Searching),
                  17 (Unique / Set Operations)
Progress:         17 / 75 chapters
Completed:        Ch1-14
Remaining:        58 chapters
```

Grouped because all three are "data reality" chapters — real datasets have missing values,
need ranking, and have duplicates/categories that need set-style logic.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 15 — NaN and Infinity

## Concept
`np.nan` (Not a Number) and `np.inf`/`-np.inf` are special floating-point values defined by the
IEEE 754 standard. They are **not** Python `None` — they are actual float values that
propagate through arithmetic in specific, well-defined (if surprising) ways.

## Why it matters
Missing sensor readings, failed API calls returning null, division-by-zero from a rare-category
ratio feature, log(0) from a feature engineering step — all become `NaN`/`inf` in real
pipelines, and if unhandled, silently poison every downstream computation.


In [ ]:
x = np.array([1.0, np.nan, 3.0, np.inf, -np.inf, 5.0])
print("x:", x)
print()
print("x == np.nan          :", x == np.nan, "  <- NaN is never equal to anything, even itself!")
print("np.isnan(x)           :", np.isnan(x))
print("np.isinf(x)           :", np.isinf(x))
print("np.isfinite(x)        :", np.isfinite(x))
print()
print("x.sum()               :", x.sum(), "  <- ONE NaN poisons the entire sum")
print("np.nansum(x)          :", np.nansum(x), "  <- ignores the NaN element, but inf + (-inf) still gives nan!")
print("np.nan_to_num(x)      :", np.nan_to_num(x), "  <- replaces nan->0, inf->huge finite number")


### Expected Output
```
x: [  1.  nan   3.  inf -inf   5.]

x == np.nan          : [False False False False False False]  <- NaN is never equal to anything, even itself!
np.isnan(x)           : [False  True False False False False]
np.isinf(x)           : [False False False  True  True False]
np.isfinite(x)        : [ True False  True False False  True]

x.sum()               : nan  <- ONE NaN poisons the entire sum
np.nansum(x)          : nan  <- ignores the NaN element, but inf + (-inf) still gives nan!
np.nan_to_num(x)      : [ 1.00000000e+000  0.00000000e+000  3.00000000e+000  1.79769313e+308
 -1.79769313e+308  5.00000000e+000]  <- replaces nan->0, inf->huge finite number
```

### The rule that trips up beginners forever
`np.nan == np.nan` is `False`. By IEEE 754 definition, NaN compares unequal to *everything*,
including itself. **Never** write `x == np.nan` to detect missing values — always use
`np.isnan(x)`.

### NaN propagation vs Infinity propagation
- Any arithmetic operation involving `NaN` produces `NaN` (it "poisons" the result).
- `np.nansum`/`np.nanmean`/etc. skip individual `NaN` *values*, but do **not** protect you from
  `inf` arithmetic — here `nansum` correctly drops the `NaN` element, but `inf + (-inf)` in the
  remaining values still evaluates to `NaN` (IEEE 754: infinity minus infinity is undefined).
  You typically need `np.isfinite()` to filter out both `NaN` *and* `inf` up front, rather than
  relying on `nan*` functions to save you from infinities.

### Common real-world source of `inf`
```python
ratio = count_a / count_b          # count_b == 0 for some rows -> inf, not an error, no exception!
log_feature = np.log(feature)      # feature contains 0 -> -inf; feature contains negative -> nan
```
`0.0 / 0.0` produces `nan`; `nonzero / 0.0` produces `inf` — NumPy's default float division does
**not** raise `ZeroDivisionError` like plain Python does; it emits a `RuntimeWarning` and
continues with `inf`/`nan`. This "fails silently, not loudly" behavior is exactly why explicit
`np.isnan`/`np.isfinite` checks after any division/log/sqrt step are standard defensive
practice in production numerical code.

### Best Practice
```python
mask = np.isfinite(x)
clean_x = x[mask]                     # drop non-finite values
# or, for a feature matrix, drop/flag ROWS with any non-finite value:
row_mask = np.isfinite(X).all(axis=1)
```

### Interview Insight
> "How do you correctly check for NaN values in a NumPy array?"
`np.isnan(x)`, never `x == np.nan` — because NaN never equals anything by IEEE 754 definition,
including itself, so `==` always yields `False` for every element, silently finding zero NaNs
even when many exist.


---
# CHAPTER 16 — Sorting and Searching

## Concept
`np.sort()` returns sorted **values**; `np.argsort()` returns the **indices** that would sort
the array (far more broadly useful, since you can use those indices to reorder *other* related
arrays in sync). `np.partition()`/`np.argpartition()` do a partial, cheaper "top-K" style sort.

## Why it matters
Ranking search results, finding the K nearest neighbors, sorting predictions by confidence
while keeping track of which sample each belongs to — nearly always you need `argsort`, not
`sort`, because you need to preserve the correspondence to other arrays (labels, IDs, ...).


In [ ]:
scores = np.array([88, 45, 97, 62, 71])
names_idx = np.array([0, 1, 2, 3, 4])  # imagine these correspond to student IDs

print("scores            :", scores)
print("np.sort(scores)    :", np.sort(scores), "  <- just the sorted VALUES, lost which student had which score")
print("np.argsort(scores) :", np.argsort(scores), "  <- INDICES that would sort scores ascending")

order = np.argsort(scores)[::-1]   # descending order of indices
print("ranked student IDs (highest score first):", names_idx[order])
print("ranked scores    (highest first)        :", scores[order])


### Expected Output
```
scores            : [88 45 97 62 71]
np.sort(scores)    : [45 62 71 88 97]  <- just the sorted VALUES, lost which student had which score
np.argsort(scores) : [1 3 4 0 2]  <- INDICES that would sort scores ascending

ranked student IDs (highest score first): [2 0 4 3 1]
ranked scores    (highest first)        : [97 88 71 62 45]
```

### `sort` vs `argsort` vs `partition` vs `argpartition`
| Function | Returns | Cost | Use when |
|---|---|---|---|
| `sort` | full sorted values | O(n log n) | you only need the values, order doesn't need to map back |
| `argsort` | full sorted indices | O(n log n) | you need to reorder *other* related arrays too (the common case) |
| `partition(k)` | values, with the k-th smallest in its final sorted position, everything smaller before it, everything larger after it (rest unordered) | O(n) average | you only need the top/bottom K, don't care about full order |
| `argpartition(k)` | indices version of `partition` | O(n) average | top-K indices, don't care which order within the K |

### Efficient Top-K (the real production pattern)


In [ ]:
rng = np.random.default_rng(0)
values = rng.integers(0, 1000, size=20)
print("values:", values)

K = 5
# Full sort approach -- O(n log n), wasteful if n is huge and K is small
top_k_full_sort = np.sort(values)[-K:][::-1]
print("top-K via full sort (descending):", top_k_full_sort)

# Efficient approach -- O(n) average via argpartition, then sort only the K results
top_k_idx = np.argpartition(values, -K)[-K:]          # K largest, unordered among themselves
top_k_idx_sorted = top_k_idx[np.argsort(values[top_k_idx])[::-1]]  # order just the K
print("top-K via argpartition (descending):", values[top_k_idx_sorted])


### Why `argpartition` matters for real ML systems
For a recommendation system scoring millions of items and returning the top 10, sorting all N
scores just to take the top 10 is wasteful — `argpartition` gets you the top-K set in O(n)
average time, and you only pay the O(K log K) sorting cost on the tiny K-sized result. This
exact pattern appears in nearest-neighbor search (Chapter 41), recommendation ranking
(Chapter 49), and any "top-K of a huge candidate set" production system.

### `np.searchsorted()` — binary search on a sorted array
```python
sorted_arr = np.array([1, 3, 5, 7, 9])
np.searchsorted(sorted_arr, 6)   # -> 3  (index where 6 would be inserted to keep sorted order)
```
Useful for binning continuous values into discrete buckets (e.g., converting a continuous
feature into categorical bins defined by boundary edges) in O(log n) per lookup.

### Common Mistake
Using `np.sort()` when you actually need to keep track of which original row/sample each sorted
value came from — losing that correspondence is a very easy mistake that silently breaks
downstream logic (e.g., attaching the wrong label to the wrong score).

### Interview Insight
> "How would you efficiently get the top-10 highest-confidence predictions from 10 million
> model outputs?"
`np.argpartition(scores, -10)[-10:]` to get the top-10 indices in O(n) average time, then
`argsort` just those 10 values if you need them fully ordered — avoiding an O(n log n) full sort
of 10 million elements when you only need 10.


---
# CHAPTER 17 — Unique / Set Operations

## Concept
`np.unique()` and friends (`isin`, `intersect1d`, `union1d`, `setdiff1d`, `setxor1d`) treat
arrays as mathematical sets, with `unique` additionally supporting counts and index mappings —
this is the array-native equivalent of `pandas.Series.unique()`/`value_counts()`, and the
foundation of label encoding.

## Why it matters
Categorical preprocessing (label encoding), deduplication, dataset analysis (how many distinct
classes? how balanced?), and set comparisons between train/test splits (data leakage checks).


In [ ]:
labels = np.array(["cat", "dog", "cat", "bird", "dog", "cat"])

unique_vals = np.unique(labels)
unique_vals2, counts = np.unique(labels, return_counts=True)
unique_vals3, first_idx = np.unique(labels, return_index=True)
unique_vals4, inverse = np.unique(labels, return_inverse=True)

print("unique labels        :", unique_vals)
print("counts per label      :", dict(zip(unique_vals2, counts)))
print("index of first occurrence:", dict(zip(unique_vals3, first_idx)))
print("inverse (label-encoded):", inverse, "  <- this IS label encoding: maps each original element to its index in unique_vals")
print("reconstruct original from inverse:", unique_vals4[inverse])


### Expected Output
```
unique labels        : ['bird' 'cat' 'dog']
counts per label      : {'bird': 1, 'cat': 3, 'dog': 2}
index of first occurrence: {'bird': 3, 'cat': 0, 'dog': 1}
inverse (label-encoded): [1 2 1 0 2 1]  <- this IS label encoding: maps each original element to its index in unique_vals
reconstruct original from inverse: ['cat' 'dog' 'cat' 'bird' 'dog' 'cat']
```
*(Note: depending on your NumPy version, the printed dict values may show as `np.int64(3)`
instead of plain `3` — this is just NumPy's scalar repr, the values themselves are unchanged.)*

### `return_inverse` IS label encoding
`np.unique(labels, return_inverse=True)` gives you both the sorted set of distinct categories
*and* an integer array mapping each original element to its category's index — this is exactly
what `sklearn.preprocessing.LabelEncoder` does internally. Understanding this means you never
need to treat that class as a black box.

### Set operations between two arrays


In [ ]:
train_ids = np.array([1, 2, 3, 4, 5, 6, 7])
test_ids  = np.array([6, 7, 8, 9])

print("intersect1d (potential leakage!) :", np.intersect1d(train_ids, test_ids))
print("union1d                          :", np.union1d(train_ids, test_ids))
print("setdiff1d (train only)           :", np.setdiff1d(train_ids, test_ids))
print("setxor1d (in exactly one set)    :", np.setxor1d(train_ids, test_ids))
print("isin (which test_ids are in train):", np.isin(test_ids, train_ids))


### Expected Output
```
intersect1d (potential leakage!) : [6 7]
union1d                          : [1 2 3 4 5 6 7 8 9]
setdiff1d (train only)           : [1 2 3 4 5]
setxor1d (in exactly one set)    : [1 2 3 4 5 8 9]
isin (which test_ids are in train): [ True  True False False]
```

### Real-world use: data leakage detection
```python
leaked_ids = np.intersect1d(train_ids, test_ids)
assert len(leaked_ids) == 0, f"Data leakage detected! {len(leaked_ids)} IDs in both train and test"
```
This exact one-liner is a standard sanity check in any serious ML pipeline before training —
catching an overlap between train/test IDs here can save you from silently inflated, unreliable
validation metrics.

### Common Mistake
Assuming `np.unique()` preserves the **original order** of first appearance — it doesn't; it
returns values in **sorted** order by default. If you need first-appearance order, use
`return_index=True` and sort the indices, or use `pandas.unique()` instead (which does preserve
order).

### Interview Insight
> "How would you check for data leakage between train and test ID arrays before training a
> model?"
`np.intersect1d(train_ids, test_ids)` — if non-empty, those IDs appear in both splits, which is
a leakage bug that can silently produce misleadingly optimistic validation metrics.


---
## Chapters 15–17 Summary

- **NaN/Infinity**: `NaN != NaN` — always use `np.isnan()`, never `== np.nan`. Division and log
  operations produce `inf`/`nan` **silently** (warning, not exception) — `np.isfinite()` is the
  standard defensive check after any division/log/sqrt in a pipeline.
- **Sorting/Searching**: prefer `argsort` over `sort` whenever you need to keep values in sync
  with other related arrays (labels, IDs). `argpartition` gives O(n) average top-K selection —
  don't full-sort millions of items just to grab the top 10.
- **Unique/Set ops**: `np.unique(..., return_inverse=True)` *is* label encoding under the hood.
  `intersect1d`/`setdiff1d`/`isin` are the standard tools for train/test leakage checks and
  categorical set comparisons. `np.unique` sorts by default — it does not preserve
  first-appearance order.

---

## Mastery Exercises

**Exercise 1 (Prediction):** Given `x = np.array([1.0, 0.0, -1.0]) / np.array([2.0, 0.0, 0.0])`,
predict each resulting value (finite, `inf`, `-inf`, or `nan`) before running it.

**Exercise 2:** You have `predictions` (model confidence scores) and `sample_ids` (parallel
array of IDs). Write the one-line expression to get `sample_ids` ordered by **descending**
confidence.

**Coding Challenge:** Implement `top_k_with_ids(scores, ids, k)` that returns the top-k
`(id, score)` pairs sorted by descending score, using `argpartition` for the initial selection
(not a full sort) for efficiency on large inputs. Then implement `check_leakage(train_ids,
test_ids)` that raises a clear `ValueError` (with the count and a sample of the overlapping IDs)
if any leakage is found.

**Debugging Challenge:**
```python
ratios = counts_a / counts_b        # counts_b has some zeros
cleaned = ratios[ratios != np.nan]  # teammate tries to drop NaNs this way -- doesn't work!
print(cleaned.shape, ratios.shape)  # same shape -- nothing was dropped
```
Explain exactly why this fails silently, and give the correct one-line fix.

**Interview Questions:**
1. Why does `x == np.nan` always return `False`, and what should you use instead?
2. What's the difference between `np.sort()` and `np.argsort()`, and why is `argsort` usually
   the more broadly useful one in real pipelines?
3. How would you efficiently get the top-100 scores out of 50 million, without a full sort?
4. How does `np.unique(arr, return_inverse=True)` relate to label encoding?

**Quiz:**
- Q1: True/False — `np.nansum()` also ignores `inf` values, the way it ignores `NaN`.
- Q2: Which function would you use to check, in O(n) average time, whether array `a` and array
  `b` share any elements, without fully sorting either?
- Q3: Does `np.unique()` preserve the order elements first appeared in the original array?

---


